# CMax V1 (`thesis_cmax`) — full CMax solve per frame as the R anchor (walkthrough)

**What this unrolls**, in the order the harness runs it (`evaluation.py::experiment_tracking`, V1 branch):

| step | code |
|---|---|
| run config, frame-0 R from gyro | `run_config.py::RunConfig` |
| event frames `V` | `interacting_maps/data_loader.py::EventFrameSequence` |
| network build | `run_config.py::make_network` → `InteractingMapsThesis.__init__`, `initialize_from_rotation` |
| raw undistorted events for CMax | `data_loader.py::load_events_fast`, `undistort_events` |
| **CMax solve** per frame | `interacting_maps/angular_velocity.py::CMaxAngularVelocity.estimate` (bearings → warp → bilinear IWE → blurred variance → Nelder-Mead) |
| **anchor** R toward ω_cmax | `network_dissertation.py::InteractingMapsThesis.step` with `omega_imu=ω_cmax` → `Cost_IMU` |
| multi-frame loop + metrics | `evaluation.py::experiment_tracking`, `metrics.py::compute_metrics` |

Every unrolled block ends with an assertion against the real module function.
Inputs: `data/poster_rotation/` (seg_C). Outputs: figures + one tracking run under `experiments/cmax_walkthrough/`.

**In V1, CMax and the network are decoupled:** CMax produces ω_cmax from events alone (warm-started from its own
previous answer, not from the network's R), and the network only feels it through `Cost_IMU`, a soft pull
`R ← R − δ_IMU·(R − ω_cmax·dt)`, which competes with `Cost_Kinematics` pulling R toward the flow-implied `R*`.

In [ ]:
# setup — run from the repo root (data/ and experiments/ are relative paths)
import os, sys
os.environ.setdefault('IM_MLFLOW', '0')          # walkthrough runs never go to MLflow
ROOT = os.path.abspath('.')
assert os.path.exists(os.path.join(ROOT, 'config.py')), 'start the notebook from the repo root'
sys.path.insert(0, ROOT)
# local slice switches (env): CMAX_WT_NFRAMES (default 5), CMAX_WT_NITERS (default ITERS_PER_FRAME), CMAX_WT_K (frame studied in detail)

In [ ]:
# imports — only what the cells use, grouped by module
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from config import ITERS_PER_FRAME
from run_config import RunConfig, make_network
from interacting_maps.data_loader import (EventFrameSequence, CameraCalibration,
                                          load_events_fast, undistort_events, events_to_vframe)
from interacting_maps.eval_io import (load_imu, load_groundtruth, load_omega_gt,
                                      get_gyro_for_frame, get_reference_omega)
from interacting_maps.metrics import compute_metrics
from interacting_maps.angular_velocity import CMaxAngularVelocity
from interacting_maps.network_dissertation import InteractingMapsThesis
from interacting_maps.camera import CLIP_I, CLIP_G, CLIP_F, CLIP_R
from interacting_maps.style import use_style, save, to_deg_s, OMEGA_AXIS_COLORS, SCREEN_WIDTH
from interacting_maps.viz import draw_map, map_scale
import evaluation as E   # only for the final end-to-end check (experiment_tracking)

In [ ]:
# constants — the walkthrough's knobs (mirror tests/test_cmax_*.py: poster seg_C, ω_x-dominant, |ω|≈1.1 rad/s)
DATASET    = 'poster_rotation'
SEGMENT_ID = 'seg_C'                                         # config.DATASET_SEGMENTS
MODEL      = 'thesis_cmax'
N_FRAMES   = int(os.environ.get('CMAX_WT_NFRAMES', 5))       # segment default is 150
N_ITERS    = int(os.environ.get('CMAX_WT_NITERS', ITERS_PER_FRAME))   # config.ITERS_PER_FRAME = 75
K          = int(os.environ.get('CMAX_WT_K', 0))             # frame studied step by step
OUT_ROOT   = 'experiments/cmax_walkthrough'                  # NOT results/ (load-bearing)
FIG_DIR    = os.path.join(OUT_ROOT, 'v1_figures')
os.makedirs(FIG_DIR, exist_ok=True)
np.set_printoptions(precision=4, suppress=True)

## 1) Run configuration — `run_config.py::RunConfig`
`RunConfig(model='thesis_cmax')` decides which costs are active and where the frame-0 `R` comes from.
For V1: `use_imu=True` (the `Cost_IMU` anchor is active — its *target* is swapped from the gyro to
ω_cmax), `use_cmax=True` (load raw events), `use_cmax_v2=False`.
`initial_R` is computed in `RunConfig._compute_initial_R`: the **gyro** mean over the first frame window,
times `dt` (rad/frame). This is the only place the IMU enters a CMax run.

In [ ]:
# run_config.py::RunConfig.__init__
rc = RunConfig(dataset=DATASET, model=MODEL, segment=SEGMENT_ID,
               n_frames=N_FRAMES, n_iters=N_ITERS, out_root=OUT_ROOT)
rc

In [ ]:
# the switches this model sets, the deltas, and the frame-0 R (rad/frame and °/s)
print({k: getattr(rc, k) for k in ('use_thesis', 'use_imu', 'use_cmax', 'use_cmax_v2',
                                   'cmax_lr', 'distortion_mode', 'poisson', 'frame_duration')})
print('params     :', rc.params)
print('initial_R  :', rc.initial_R, 'rad/frame  =', to_deg_s(rc.initial_R / rc.frame_duration), '°/s')
print('output_dir :', rc.output_dir)

## 2) Event frames `V` — `interacting_maps/data_loader.py::EventFrameSequence`
The network's input. Events are read once for `[t_start, t_start + n·dt + 0.1 s)`; each frame bins its window
into a signed count image, clipped at ±10 and divided by 10 (`clip_value=10.0` in `experiment_tracking`).
In `C_full` mode the events are **not** undistorted here (lens handled inside the kinematic matrix `C`).

In [ ]:
# evaluation.py::experiment_tracking — EventFrameSequence(...) exactly as the harness builds it
seq = EventFrameSequence(
    rc.paths['events'], rc.paths['calib'],
    frame_duration=rc.frame_duration, t_start=rc.t_start,
    n_frames=rc.n_frames, clip_value=10.0,
    undistort=rc.undistort_at_event_level,
    sensor_size=rc.sensor_size,
)
H, W = seq.H, seq.W
fx, fy, cx, cy = seq.calib.fx, seq.calib.fy, seq.calib.cx, seq.calib.cy
frames = list(seq)                         # [(V, t_mid), ...]
print('undistort at event level:', seq.undistort, '| dist coeffs:', seq.calib.dist)

In [ ]:
# what V looks like: per frame, events binned, share of saturated pixels (|V| = 1 ↔ ≥10 net events)
pd.DataFrame([{'frame': k, 't_mid': t, 'nonzero_px': int((V != 0).sum()),
               'saturated_px': int((np.abs(V) >= 1).sum()), 'V_min': V.min(), 'V_max': V.max()}
              for k, (V, t) in enumerate(frames)])

In [ ]:
# sensors used to SCORE (Vicon, independent) and the gyro (frame-0 init + cross-check only)
imu_data = load_imu(rc.paths['imu'])
gt_data  = load_groundtruth(rc.paths['groundtruth'])
om_data  = load_omega_gt(rc.paths.get('omega_gt'))
print('imu rows', len(imu_data), '| gt rows', len(gt_data), '| omega_gt', om_data is not None)

## 3) Network — `run_config.py::make_network` (thesis branch, unrolled)
`C_full`: the network gets the lens coefficients so `C` is distortion-aware. Then `initialize_from_rotation`
sets `R = initial_R`, `F = C·R` (mutually consistent start) and `I` = 0.01·N(0,1) with seed 42; `G = 0`.
For V1 nothing CMax-specific happens here — the CMax estimator lives outside the network.

In [ ]:
# run_config.py::make_network — dist_coeffs for C_full
dist_coeffs = CameraCalibration(rc.paths['calib']).dist if rc.distortion_mode == 'C_full' else None
dist_coeffs

In [ ]:
# run_config.py::make_network — InteractingMapsThesis(...) + initialize_from_rotation(initial_R)
net = InteractingMapsThesis(H=H, W=W, fx=fx, fy=fy, cx=cx, cy=cy,
                            frame_duration=rc.frame_duration, dist_coeffs=dist_coeffs,
                            poisson=rc.poisson, **rc.params)
net.initialize_from_rotation(rc.initial_R)
print('costs:', [type(c).__name__ for c in net.costs], '| r_update:', net.r_update,
      '| kinematics updates R:', net.cost_kin.update_r)

In [ ]:
# the starting state: F = C·R exactly, I tiny noise, G zero
print('|F - C·R| max :', np.abs(net.F - np.einsum('hwij,j->hwi', net._C_mat, net.R)).max())
print('F px/frame    : mean |F| =', np.hypot(net.F[..., 0], net.F[..., 1]).mean())
print('I std, G max  :', net.I.std(), np.abs(net.G).max())

In [ ]:
# check: make_network == unrolled build
net_chk = make_network(rc, H, W, fx, fy, cx, cy)
for q in ('q_I', 'q_G', 'q_F', 'q_R'):
    assert np.array_equal(getattr(net, q).value, getattr(net_chk, q).value), q
assert [type(c) for c in net.costs] == [type(c) for c in net_chk.costs]
print('[ok] make_network == unrolled build')

## 4) Events for CMax — `load_events_fast` + `undistort_events` (`evaluation.py::experiment_tracking`, "B1 plumbing")
CMax does not see `V`; it gets the **raw event list**, re-loaded over the same span and **undistorted** with
`cv2.undistortPoints` (pixel coordinates become floats, same `K`). So in `C_full` mode the network works on
distorted pixels with a distortion-aware `C`, while CMax works on undistorted pixels with a pure pinhole warp —
two representations of the same geometry. Points that undistort outside the sensor are kept here; they are only
dropped later when they are voted into the IWE.

In [ ]:
# evaluation.py::experiment_tracking — dur = n_frames·dt + 0.1, then undistort ONCE
dur = rc.n_frames * rc.frame_duration + 0.1
raw_events = load_events_fast(rc.paths['events'], t_start=rc.t_start, duration=dur)
cmax_events = undistort_events(raw_events, seq.calib)
print('events:', len(cmax_events), '| same span as EventFrameSequence:',
      np.array_equal(raw_events, seq._events))

In [ ]:
# what undistortion does to the coordinates: shift size and how many leave the sensor
shift = np.hypot(cmax_events[:, 1] - raw_events[:, 1], cmax_events[:, 2] - raw_events[:, 2])
outside = ((cmax_events[:, 1] < 0) | (cmax_events[:, 1] > W - 1) |
           (cmax_events[:, 2] < 0) | (cmax_events[:, 2] > H - 1))
print(f'shift px: median {np.median(shift):.2f}, p99 {np.percentile(shift, 99):.2f}, max {shift.max():.2f}')
print(f'outside the sensor after undistortion: {outside.mean():.2%}')
print('polarity values:', np.unique(cmax_events[:, 3]))

## 5) Frame window `K` — `evaluation.py::experiment_tracking` (per-frame slice)
The harness slices `[t_lo, t_hi)` out of the undistorted list for each frame. Below everything is done for one
frame `K` (default 0). V1 passes `t_ref = 0.5·(t_lo+t_hi)` (frame centre) to `estimate`; V2 uses the event-span midpoint.

In [ ]:
# evaluation.py::experiment_tracking — window bounds, slice, gyro (input) and reference (scoring)
t_lo = rc.t_start + K * rc.frame_duration
t_hi = t_lo + rc.frame_duration
V, t_mid = frames[K]
win = cmax_events[(cmax_events[:, 0] >= t_lo) & (cmax_events[:, 0] < t_hi)]
omega_imu = get_gyro_for_frame(imu_data, t_lo, t_hi)
omega_ref, ref_src = get_reference_omega(gt_data, imu_data, t_lo, t_hi, om_data)
print(f'frame {K}: [{t_lo:.4f}, {t_hi:.4f})  events {len(win)}  ON share {win[:, 3].mean():.2%}')
print('ω gyro  °/s :', to_deg_s(omega_imu))
print(f'ω ref   °/s : {to_deg_s(omega_ref)}   ({ref_src})')

In [ ]:
# undistortion before/after on the FIRST frame (frame 0, independent of K): same events, raw vs undistorted
# coordinates, binned nearest-pixel with the pipeline's events_to_vframe (clip 10, as EventFrameSequence)
t0_lo, t0_hi = rc.t_start, rc.t_start + rc.frame_duration
m0 = (raw_events[:, 0] >= t0_lo) & (raw_events[:, 0] < t0_hi)     # undistort_events keeps row order
ev_raw, ev_und = raw_events[m0].copy(), cmax_events[m0].copy()
ev_und[:, 1:3] = np.floor(ev_und[:, 1:3] + 0.5)                   # round; events_to_vframe truncates toward 0
V_raw = events_to_vframe(ev_raw, H, W, clip_value=10.0)
V_und = events_to_vframe(ev_und, H, W, clip_value=10.0)

# lens displacement field: undistort a pixel grid (dummy t/pol columns) and take undistorted - raw
gy, gx = np.mgrid[0:H:12, 0:W:12].astype(np.float64)
grid = np.stack([np.zeros(gx.size), gx.ravel(), gy.ravel(), np.ones(gx.size)], axis=1)
grid_und = undistort_events(grid.copy(), seq.calib)
dxy = grid_und[:, 1:3] - grid[:, 1:3]
yy, xx = np.mgrid[0:H, 0:W].astype(np.float64)
dense = undistort_events(np.stack([np.zeros(xx.size), xx.ravel(), yy.ravel(), np.ones(xx.size)], axis=1), seq.calib)
shift_map = np.hypot(dense[:, 1] - xx.ravel(), dense[:, 2] - yy.ravel()).reshape(H, W)
n_out = ((ev_und[:, 1] < 0) | (ev_und[:, 1] >= W) | (ev_und[:, 2] < 0) | (ev_und[:, 2] >= H)).sum()
print(f'frame 0: [{t0_lo:.4f}, {t0_hi:.4f})  events {m0.sum()}  outside the sensor after undistortion {n_out}')
print(f'pixel shift on the sensor grid: centre {shift_map[H // 2, W // 2]:.2f} px, corner max {shift_map.max():.2f} px')

s = map_scale('V', V_raw)
with use_style('screen'):
    fig, axes = plt.subplots(1, 3, figsize=(SCREEN_WIDTH, 2.6))
    draw_map(axes[0], 'V', V_raw, title='raw events (frame 0)', scale=s)
    draw_map(axes[1], 'V', V_und, title='undistorted events (frame 0)', scale=s)
    im = axes[2].imshow(shift_map, cmap='viridis', interpolation='nearest')
    axes[2].quiver(gx.ravel(), gy.ravel(), dxy[:, 0], dxy[:, 1], color='w', angles='xy', width=0.004)
    axes[2].set_xticks([]); axes[2].set_yticks([]); axes[2].set_title('undistortion shift (px)')
    fig.colorbar(im, ax=axes[2], fraction=0.035, pad=0.02)
    path = os.path.join(FIG_DIR, 'v1_undistortion_frame0.png'); save(fig, path)
display(Image(path))

In [ ]:
# the two candidate reference times: frame centre (V1 passes this) vs event-span midpoint (V2 / estimate default)
t_ref_frame  = 0.5 * (t_lo + t_hi)
t_ref_events = 0.5 * (win[:, 0].min() + win[:, 0].max())
print(f'frame centre {t_ref_frame:.6f} s | event midpoint {t_ref_events:.6f} s | '
      f'difference {(t_ref_events - t_ref_frame) * 1e3:+.3f} ms')

## 6) CMax estimator — `angular_velocity.py::CMaxAngularVelocity.__init__`
Built in `experiment_tracking` with `use_polarity=True` and the defaults `blur_sigma=1.0`,
`optimizer='Nelder-Mead'`, `use_analytic_gradient=False`, **`dist_coeffs=None`** → pinhole warp on the
already-undistorted events ("Way 1"). The estimator keeps no state between calls except `last_iwe`/`last_result`;
the warm start is passed in by the caller.

In [ ]:
# evaluation.py::experiment_tracking — the V1 estimator
cmax_est = CMaxAngularVelocity(H, W, fx, fy, cx, cy, use_polarity=True)
{k: getattr(cmax_est, k) for k in ('use_polarity', 'blur_sigma', 'optimizer', 'use_analytic_gradient', 'dist')}

## 7) `CMaxAngularVelocity.estimate` — unrolled
### 7a) Inputs → per-event arrays
`dt = t − t_ref` is each event's signed time offset (s) from the reference time — events are warped *to* `t_ref`.
Fewer than 10 events would return the warm start unchanged.

In [ ]:
# angular_velocity.py::estimate — the warm start the harness passes: omega_cmax_prev, zeros on the first frame
omega_cmax_prev = np.zeros(3)            # evaluation.py: NOT the gyro-initialised network R
omega_init = omega_cmax_prev
t_ref = t_ref_frame                      # evaluation.py passes t_ref=0.5*(t_lo+t_hi)
assert len(win) >= 10                    # estimate(): early-return guard

In [ ]:
# angular_velocity.py::estimate — split columns, time offsets
t   = win[:, 0].astype(np.float64)
xs  = win[:, 1].astype(np.float64)
ys  = win[:, 2].astype(np.float64)
pol = win[:, 3].astype(np.float64)
dt  = t - t_ref
print(f'dt range [{dt.min()*1e3:+.2f}, {dt.max()*1e3:+.2f}] ms, mean {dt.mean()*1e3:+.3f} ms')

### 7b) Bearings — `CMaxAngularVelocity._bearings`
`dist=None`: `p = normalize(((x−cx)/fx, (y−cy)/fy, 1))`, a unit ray per event. The rotation acts on these rays.

In [ ]:
# angular_velocity.py::_bearings — pinhole back-projection to unit rays
bearings = cmax_est._bearings(xs, ys)
ang = np.degrees(np.arccos(bearings[:, 2]))
print('shape', bearings.shape, '| |p| range', np.linalg.norm(bearings, axis=1).min(), np.linalg.norm(bearings, axis=1).max())
print(f'angle off optical axis: median {np.median(ang):.1f}°, max {ang.max():.1f}°')

### 7c) Weights — polarity
`use_polarity=True` → `w = 2·pol − 1 ∈ {−1, +1}`. ON and OFF events cancel where they overlap in the IWE.

In [ ]:
# angular_velocity.py::estimate — signed polarity weights
weights = (2.0 * pol - 1.0) if cmax_est.use_polarity else np.ones_like(pol)
print(pd.Series(weights).value_counts().to_dict(), '| Σw =', weights.sum())
x0 = np.zeros(3) if omega_init is None else np.asarray(omega_init, float).copy()
print('x0 (warm start) °/s:', to_deg_s(x0))

### 7d) Warp — `CMaxAngularVelocity._warp_to_pixels`
Linear (first-order) rotation `p_rot = p + (ω·dt) × p`, then pinhole re-projection. At ω=0 the warp is the
identity. The displacement grows linearly with |dt| (max ±10 ms) and |ω| — this is how far CMax moves each event.

In [ ]:
# angular_velocity.py::_warp_to_pixels — at the warm start and at the gyro ω
xw0, yw0 = cmax_est._warp_to_pixels(bearings, dt, x0)
xwg, ywg = cmax_est._warp_to_pixels(bearings, dt, omega_imu)
disp0 = np.hypot(xw0 - xs, yw0 - ys); dispg = np.hypot(xwg - xs, ywg - ys)
print(f'|displacement| at x0   : median {np.median(disp0):.3f} px, max {disp0.max():.3f} px')
print(f'|displacement| at gyro : median {np.median(dispg):.3f} px, max {dispg.max():.3f} px')

### 7e) IWE — `CMaxAngularVelocity._accumulate_iwe` (bilinear voting)
Each warped event spreads its weight over the 4 neighbouring pixels. Votes that land outside the sensor are
**dropped**, so `Σ IWE` can be smaller than `Σ w` — which events count depends on ω, so the score can move for a reason unrelated to sharpness (see cmax_todo.md, border handling).

In [ ]:
# angular_velocity.py::_accumulate_iwe — IWE at x0 and at the gyro ω
iwe_x0   = cmax_est._accumulate_iwe(xw0, yw0, weights)
iwe_gyro = cmax_est._accumulate_iwe(xwg, ywg, weights)
for name, iwe in (('x0', iwe_x0), ('gyro', iwe_gyro)):
    print(f'{name:5s} ΣIWE {iwe.sum():9.1f} (Σw {weights.sum():.1f})  Σ|IWE| {np.abs(iwe).sum():9.1f}  var {iwe.var():.5f}')

### 7f) Objective — `CMaxAngularVelocity._contrast`
`Var(gaussian_filter(IWE, σ=1))` — the **blurred** variance, maximized. Compare the value at the warm start,
the gyro and the reference: the true ω should give the highest contrast.

In [ ]:
# angular_velocity.py::_contrast — blurred-IWE variance at three candidate ω
for name, w in (('x0', x0), ('gyro', omega_imu), ('ref', omega_ref)):
    print(f'{name:5s} ω={to_deg_s(w)} °/s  contrast={cmax_est._contrast(w, bearings, dt, weights):.6f}')

### 7g) Optimizer — Nelder-Mead on −contrast
Initial simplex = `x0` plus `0.5 rad/s` (≈29 °/s) along each axis; stop at `xatol=1e-3` rad/s, `fatol=1e-9`,
`maxiter=400`. Derivative-free, deterministic.

In [ ]:
# angular_velocity.py::estimate — Nelder-Mead branch
from scipy.optimize import minimize
neg = lambda w: -cmax_est._contrast(w, bearings, dt, weights)
span = 0.5
init_simplex = np.vstack([x0, x0 + span * np.eye(3)])
res = minimize(neg, x0, method='Nelder-Mead',
               options=dict(initial_simplex=init_simplex, xatol=1e-3, fatol=1e-9, maxiter=400))
print(res.message, '| nit', res.nit, '| nfev', res.nfev)
print('ω_cmax °/s:', to_deg_s(res.x), '| contrast', -res.fun)

In [ ]:
# angular_velocity.py::estimate — the diagnostic IWE it keeps (UNblurred, at the solution)
last_iwe = cmax_est._build_iwe(bearings, dt, weights, res.x)
omega_anchor = res.x
for name, w in (('gyro', omega_imu), ('ref', omega_ref)):
    e, d, b = compute_metrics(omega_anchor, w)
    print(f'ω_cmax vs {name}: err {e:.2f} °/s, dir {d:.2f}°, β {b:.3f}')

In [ ]:
# check: CMaxAngularVelocity.estimate == unrolled steps
chk_est = CMaxAngularVelocity(H, W, fx, fy, cx, cy, use_polarity=True)
omega_chk = chk_est.estimate(win, t_ref=t_ref, omega_init=omega_cmax_prev)
assert np.array_equal(omega_chk, omega_anchor), (omega_chk, omega_anchor)
assert np.array_equal(chk_est.last_iwe, last_iwe)
print('[ok] estimate() == unrolled')

### 7h) What the optimizer saw — contrast slices and IWEs
1-D slices of the objective through ω_cmax along each axis (±60 °/s), with gyro and reference marked. A sharp,
single peak at the reference means the objective is well posed for this frame; a flat or shifted peak means
CMax can't be trusted here.

In [ ]:
# landscape: _contrast along each axis through ω_cmax (inspection only, 41 evals per axis)
offs = np.radians(np.linspace(-60, 60, 41))
with use_style('screen'):
    fig, axes = plt.subplots(1, 3, figsize=(SCREEN_WIDTH, 3))
    for a, ax in enumerate(axes):
        vals = []
        for o in offs:
            w = omega_anchor.copy(); w[a] += o
            vals.append(cmax_est._contrast(w, bearings, dt, weights))
        ax.plot(to_deg_s(omega_anchor[a] + offs), vals, color=OMEGA_AXIS_COLORS[a])
        ax.axvline(to_deg_s(omega_anchor[a]), color=OMEGA_AXIS_COLORS[a], ls='-', lw=0.8, label='CMax')
        ax.axvline(to_deg_s(omega_ref[a]), color='k', lw=1.2, label='reference')
        ax.axvline(to_deg_s(omega_imu[a]), color='0.55', ls=':', label='gyro')
        ax.set_xlabel(f'$\\omega_{"xyz"[a]}$ (°/s)')
    axes[0].set_ylabel('blurred IWE variance'); axes[0].legend(fontsize=7)
    path = os.path.join(FIG_DIR, 'v1_contrast_slices.png'); save(fig, path)
display(Image(path))

In [ ]:
# IWE at the warm start, at the gyro, and at ω_cmax (shared grey scale)
s = map_scale('IWE', last_iwe)
with use_style('screen'):
    fig, axes = plt.subplots(1, 4, figsize=(SCREEN_WIDTH, 2.6))
    draw_map(axes[0], 'V', V, title=f'V (frame {K})')
    draw_map(axes[1], 'IWE', iwe_x0, title='IWE at x0 (no warp)', scale=s)
    draw_map(axes[2], 'IWE', iwe_gyro, title='IWE at gyro ω', scale=s)
    draw_map(axes[3], 'IWE', last_iwe, title='IWE at ω_cmax', scale=s)
    path = os.path.join(FIG_DIR, 'v1_iwe.png'); save(fig, path)
display(Image(path))

## 8) Anchor ω_cmax into the network — `InteractingMapsThesis.step(V, omega_imu=ω_cmax)` (unrolled)
`step` stores `V`, sets the anchor target `Cost_IMU.R_imu = ω_cmax·dt` and runs `n_iters` two-phase Jacobi
iterations: Phase 1 every cost adds its message to the accumulators, Phase 2 `value −= accumulator`, then clip.
On `R` two messages compete each iteration:
`Cost_Kinematics`: `δ_FR·(R − R*)` with `R* = M⁻¹ΣCᵀF` (flow-implied), and `Cost_IMU`: `δ_IMU·(R − ω_cmax·dt)`.
If `R*` were frozen, `R` would settle at the weighted mean `(δ_FR·R* + δ_IMU·R_anchor)/(δ_FR+δ_IMU)`.

In [ ]:
# keep an untouched copy for the equality check
net_ref = copy.deepcopy(net)

In [ ]:
# network_dissertation.py::step — inputs and anchor target
net.q_V.value = V
net.cost_imu.R_imu = omega_anchor * net.frame_duration
net._use_imu = True
print('anchor R_imu (rad/frame):', net.cost_imu.R_imu, '| start R:', net.R)

In [ ]:
# network_dissertation.py::step — the n_iters loop, recording each cost's message on R
R_iters, R_msgs, R_star = [net.q_R.value.copy()], [], []
for _ in range(rc.n_iters):
    # PHASE 1
    for q in [net.q_I, net.q_G, net.q_F, net.q_R]:
        q.reset_gradient()
    msgs = {}
    for cost in net.costs:
        if cost is net.cost_imu and not net._use_imu:
            continue
        before = net.q_R.gradient_accumulator.copy()
        cost.compute_and_send_gradients()
        msgs[type(cost).__name__] = net.q_R.gradient_accumulator - before
    R_msgs.append(msgs)
    R_star.append(net.R - msgs['Cost_Kinematics'] / net.cost_kin.delta_FR)   # implied R* this iter
    # PHASE 2
    net.q_I.update(1.0); net.q_G.update(1.0); net.q_F.update(1.0); net.q_R.update(1.0)
    net.q_I.value = np.clip(net.q_I.value, -CLIP_I, CLIP_I)
    net.q_G.value = np.clip(net.q_G.value, -CLIP_G, CLIP_G)
    net.q_F.value = np.clip(net.q_F.value, -CLIP_F, CLIP_F)
    net.q_R.value = np.clip(net.q_R.value, -CLIP_R, CLIP_R)
    R_iters.append(net.q_R.value.copy())
R_iters, R_star = np.array(R_iters), np.array(R_star)
print('R after step °/s:', to_deg_s(net.R / rc.frame_duration))

In [ ]:
# check: InteractingMapsThesis.step == unrolled loop
net_ref.step(V, n_iters=rc.n_iters, omega_imu=omega_anchor)
for q in ('q_I', 'q_G', 'q_F', 'q_R'):
    assert np.array_equal(getattr(net, q).value, getattr(net_ref, q).value), q
print('[ok] step() == unrolled')

In [ ]:
# who moved R: message sizes (°/s-equivalent) at the first/last iteration, and clip hits
for i in (0, rc.n_iters - 1):
    print(f'iter {i:3d}', {k: to_deg_s(v / rc.frame_duration).round(3) for k, v in R_msgs[i].items()
                          if np.any(v)})
print('flow-implied R* at the end  °/s:', to_deg_s(R_star[-1] / rc.frame_duration))
wm = (net.cost_kin.delta_FR * R_star[-1] + net.cost_imu.delta_IMU * net.cost_imu.R_imu) / \
     (net.cost_kin.delta_FR + net.cost_imu.delta_IMU)
print('weighted-mean prediction °/s:', to_deg_s(wm / rc.frame_duration), '| actual', to_deg_s(net.R / rc.frame_duration))
print('share at clip bound: F', np.mean(np.abs(net.F) >= CLIP_F), 'G', np.mean(np.abs(net.G) >= CLIP_G),
      'I', np.mean(np.abs(net.I) >= CLIP_I))

In [ ]:
# R over the iterations vs the anchor, flow-implied R*, gyro and reference
it = np.arange(rc.n_iters + 1)
with use_style('screen'):
    fig, axes = plt.subplots(1, 3, figsize=(SCREEN_WIDTH, 3), sharex=True)
    for a, ax in enumerate(axes):
        ax.plot(it, to_deg_s(R_iters[:, a] / rc.frame_duration), color=OMEGA_AXIS_COLORS[a], label='network R')
        ax.plot(it[1:], to_deg_s(R_star[:, a] / rc.frame_duration), color=OMEGA_AXIS_COLORS[a], ls='--', lw=0.8, label='R* (flow)')
        ax.axhline(to_deg_s(omega_anchor[a]), color='m', lw=0.9, label='anchor ω_cmax')
        ax.axhline(to_deg_s(omega_ref[a]), color='k', lw=1.2, label='reference')
        ax.axhline(to_deg_s(omega_imu[a]), color='0.55', ls=':', label='gyro')
        ax.set_title(f'$\\omega_{"xyz"[a]}$ (°/s)'); ax.set_xlabel('iteration')
    axes[0].legend(fontsize=7)
    path = os.path.join(FIG_DIR, 'v1_R_over_iterations.png'); save(fig, path)
display(Image(path))

## 9) All frames — `evaluation.py::experiment_tracking` V1 branch (unrolled)
Fresh network, then per frame: slice → `estimate(win, t_ref=frame centre, omega_init=previous ω_cmax)` →
`step(V, omega_imu=ω_cmax)` → `ω_est = R/dt` → `compute_metrics` against the reference.
The table shows how far the network's ω moves away from its own anchor.

In [ ]:
# evaluation.py::experiment_tracking — V1 loop
net = make_network(rc, H, W, fx, fy, cx, cy)
cmax_est = CMaxAngularVelocity(H, W, fx, fy, cx, cy, use_polarity=True)
omega_cmax_prev = np.zeros(3)
rows = []
for k, (V_k, t_mid_k) in enumerate(frames):
    t_lo_k = rc.t_start + k * rc.frame_duration
    t_hi_k = t_lo_k + rc.frame_duration
    omega_imu_k = get_gyro_for_frame(imu_data, t_lo_k, t_hi_k)
    omega_ref_k, _ = get_reference_omega(gt_data, imu_data, t_lo_k, t_hi_k, om_data)
    win_k = cmax_events[(cmax_events[:, 0] >= t_lo_k) & (cmax_events[:, 0] < t_hi_k)]
    omega_anchor_k = cmax_est.estimate(win_k, t_ref=0.5 * (t_lo_k + t_hi_k), omega_init=omega_cmax_prev)
    omega_cmax_prev = omega_anchor_k.copy()
    net.step(V_k, n_iters=rc.n_iters, omega_imu=omega_anchor_k)
    omega_est = net.R / rc.frame_duration
    err, dir_err, beta = compute_metrics(omega_est, omega_ref_k)
    rows.append({'frame': k, 't_mid': t_mid_k, 'n_ev': len(win_k), 'nm_nfev': cmax_est.last_result.nfev,
                 **{f'cmax_w{c}': v for c, v in zip('xyz', to_deg_s(omega_anchor_k))},
                 **{f'est_w{c}': v for c, v in zip('xyz', to_deg_s(omega_est))},
                 **{f'ref_w{c}': v for c, v in zip('xyz', to_deg_s(omega_ref_k))},
                 **{f'imu_w{c}': v for c, v in zip('xyz', to_deg_s(omega_imu_k))},
                 'err_est': err, 'err_cmax': compute_metrics(omega_anchor_k, omega_ref_k)[0],
                 'err_gyro': compute_metrics(omega_imu_k, omega_ref_k)[0],
                 'est_minus_anchor': to_deg_s(np.linalg.norm(omega_est - omega_anchor_k)),
                 'dir_err': dir_err, 'beta': beta})
df = pd.DataFrame(rows)
df.round(2)

In [ ]:
# summary (°/s): network vs its anchor vs the gyro, all against the reference
df[['err_est', 'err_cmax', 'err_gyro', 'est_minus_anchor', 'dir_err', 'beta']].describe().round(2)

In [ ]:
# check: evaluation.experiment_tracking (V1) == unrolled loop  (writes to experiments/cmax_walkthrough/)
E.experiment_tracking(rc, save_frames=False)
trk = pd.read_csv(os.path.join(rc.output_dir, 'tracking.csv'))
assert np.allclose(np.degrees(trk[['est_wx', 'est_wy', 'est_wz']].values),
                   df[['est_wx', 'est_wy', 'est_wz']].values, atol=1e-9)
assert np.allclose(trk['err_deg_s'].values, df['err_est'].values, atol=1e-9)
print('[ok] experiment_tracking == unrolled V1 loop')

In [ ]:
# ω per axis over the frames: estimate vs reference vs gyro (°/s)
t_axis = df['t_mid'] - rc.t_start
with use_style('screen'):
    fig, axes = plt.subplots(1, 3, figsize=(SCREEN_WIDTH, 3.2), sharex=True)
    for a, ax in enumerate(axes):
        ax.plot(t_axis, df[f'est_w{"xyz"[a]}'], color=OMEGA_AXIS_COLORS[a], label='network ω')
        ax.plot(t_axis, df[f'cmax_w{"xyz"[a]}'], color='m', lw=0.9, ls='--', label='anchor ω_cmax')
        ax.plot(t_axis, df[f'ref_w{"xyz"[a]}'], color='k', lw=1.2, label='reference')
        ax.plot(t_axis, df[f'imu_w{"xyz"[a]}'], color='0.55', ls=':', label='gyro')
        ax.set_title(f'$\\omega_{"xyz"[a]}$ (°/s)')
        ax.set_xlabel('time (s)')
    axes[0].legend(fontsize=7)
    path = os.path.join(FIG_DIR, 'v1_omega_over_frames.png')
    save(fig, path)
display(Image(path))

## What to check
- **7a warm start:** frame 0 starts Nelder-Mead from **zero**, not from the gyro-initialised R — does `nfev`
  on frame 0 stand out, and does ω_cmax(0) land on the same peak as later frames?
- **7e border loss:** `ΣIWE < Σw` when warped events leave the sensor; does it matter for the contrast?
- **7h landscape:** is the peak sharp and centred on the reference on every axis (ω_z is usually the weakest)?
- **8 anchor vs kinematics:** with `δ_FR=0.10`, `δ_IMU=0.50` the final R is ~5:1 anchor:flow — is R* (flow)
  pulling toward or away from the reference?
- **9 `est_minus_anchor`:** how much the network changes the CMax answer, and whether `err_est` is better or
  worse than `err_cmax` (if worse, the network only adds noise on top of CMax in V1).